# Trinidad and Tobago GDP nowcasting pipeline

Every knob is in `config.py` (its sections say which cell reads them); every code cell starts with a tag `# [tto:<name>]` so the runners find it by name.

**Two runs** share the notebook and differ only in config section 7:

| | `TTO_RUN_TAG=ident` | `TTO_RUN_TAG=prod` |
|---|---|---|
| test window | 2019-Q4 to 2022-Q4 (COVID 2020-Q2 is a test quarter) | 2023-Q2 to the last print |
| live readings | none | the unprinted quarters |
| purpose | identify the watchlist clusters | the nowcast |

**Three ways to run it**: cell by cell here (one seed, `NWCST_SEED`); `python run_tto_seed.py <seed>` (one seed, checkpointed); `python run_tto.py` (the three seeds in parallel, merged, then the post, live and chart cells).

**Cells**: 1 setup → 2 data → 3 filters → 4 model library → 5 pre-selection (four cells) → 6 specifications → 7 training loop (two cells) → 8 watchlist, regime-aware nowcast, bands → 9 live quarter.


## 1. Setup: config and paths

Loads `config.py` from this folder and derives the project folders. Nothing else is set in the notebook.


In [ ]:
# [tto:setup]
# Setup: load config.py from this folder.
# Reads:  config.py (every knob; the environment variables TTO_RUN_TAG, TTO_LAUNCH_DATE_STR and TTO_SA_METHOD select the run).
# Writes: `cfg` (the config module) and `P` (the project folders: code, raw, data, output).
# Edit config.py, not this notebook.
import os, importlib.util
CODE_DIR = os.getcwd()
_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)
print(f"[CONFIG] {cfg.COUNTRY} | run {cfg.RUN_TAG} | date_str {cfg.DATE_STR} | "
      f"train {cfg.TRAIN_START} -> test {cfg.TEST_START_ACTIVE} -> end {cfg.TEST_END_ACTIVE or 'last print'} -> live {cfg.DESIRED_DATE_ACTIVE}")


## 2. Data, dates and estimator switches

Sample dates and output file names from the config; the estimator switches (`run`); the pool filter and the reading guard that the post cells apply; the raw panel loaded and deflated with the CPI. Output folder: `output/<date>_tto_<run>/`.


In [ ]:
# [tto:data]
# Data: sample dates, output file names, estimator switches, pool filter; then the raw panel loaded and deflated.
# Reads:  cfg sections 1, 5 and 7; data/<DATA_CSV> and data/<METADATA_CSV>.
# Writes: train_start_date, test_start_date, test_end_date, desired_date, date_str, the three output file names,
#         target_variable, lags, run, _iter_, pool_mask(), reading_guard(), VINTAGES / VINTAGE_NAMES / VINTAGE_COLS,
#         PATH_OUTPUT (created), `data` (deflated levels) and `metadata`.
# =========================================================
# Run settings
# =========================================================

import pandas as pd
from datetime import datetime

# ---------------------------------------------------------
# 0) DATA/METADATA FILE NAMES
# ---------------------------------------------------------
data_file, meta_file = cfg.DATA_CSV, cfg.METADATA_CSV   # from config.py section 1

# ---------------------------------------------------------
# 1) DEFINE TRAIN / TEST WINDOW
# ---------------------------------------------------------
# These dates define the sample splits used for model training and evaluation.
# - Training sample:   [train_start_date, test_start_date)
# - Test sample:       [test_start_date, test_end_date]
train_start_date = cfg.TRAIN_START          # 2013-03-01
test_start_date  = cfg.TEST_START_ACTIVE    # prod 2023-06-01 / ident 2019-12-01

# Convert to Timestamp for safer comparisons later
train_start_date = pd.to_datetime(train_start_date)
test_start_date  = pd.to_datetime(test_start_date)

# ---------------------------------------------------------
# 2) DEFINE OUT-OF-SAMPLE FORECAST TARGET (LAST DATE / VINTAGE)
# ---------------------------------------------------------
# This is the date for which you want to produce the final nowcast/forecast.
# Use the first day of the month (or the appropriate GDP reference month).
desired_date = pd.to_datetime(cfg.DESIRED_DATE_ACTIVE) if cfg.DESIRED_DATE_ACTIVE else None   # None in the identification run

# ---------------------------------------------------------
# 3) DEFINE OUTPUT FILENAMES (timestamped)
# ---------------------------------------------------------
# Timestamp output files so each run creates a unique set of results.
date_str = cfg.DATE_STR   # the orchestrator locks it through TTO_LAUNCH_DATE_STR

forecast_file     = f"{date_str} 3_nwcst Tab fcst.xlsx"   # model forecasts / nowcasts
performance_file  = f"{date_str} 3_nwcst Tab perf.xlsx"   # model performance metrics
outofsample_file  = f"{date_str} 3_nwcst Tab oos.xlsx"    # full OOS prediction panel

# (Optional) If you want to resume / compare against a previous run,
# set last_performance to the filename of the prior performance workbook.
# Example: last_performance = "perf_20251207.xlsx"
last_performance = None

# ---------------------------------------------------------
# 4) MODEL SETTINGS
# ---------------------------------------------------------
# Target GDP series ID
target_variable = cfg.TARGET

# Prediction horizon (vintages) to evaluate:
# - negative values: backcasts (e.g., -1 = one period back)
# - 0: nowcast (current period)
# - positive values: forecasts (e.g., +1 = one period ahead)
lags = list(cfg.LAGS)

# ---------------------------------------------------------
# 5) TARGET MODELS
# ---------------------------------------------------------
# Pool filter: which estimated cells enter the regime-aware nowcast, the bands, the clusters and the live quarter.
# The tree estimator is DT4 (four-quarter leaves); DT (two-quarter leaves) is switched off in config RUN["dt"].
# The mask also drops any DT rows found in an older output, so a re-read pool is consistent.
def pool_mask(df):
    est = df["estimator"].astype(str).str.upper()
    return ~(est == "DT")

# vintage column names of the fcst table (same rule as nwcst_helpers.vintage_col_names: the -2..+2 grid keeps the legacy names)
_legacy = {-2: "two_back", -1: "one_back", 0: "zero_back", 1: "one_ahead", 2: "two_ahead"}
VINTAGE_NAMES = {l: _legacy[l] for l in lags} if sorted(lags) == [-2, -1, 0, 1, 2] else {l: f"p{l:+d}" for l in sorted(lags)}
VINTAGES = {name: l for l, name in VINTAGE_NAMES.items()}        # name -> vintage, used by the post and live cells
VINTAGE_COLS = list(VINTAGE_NAMES.values())
def reading_guard(df, cols=VINTAGE_COLS, label=""):
    """A reading beyond +/- READING_GUARD pp (config section 6) is dropped from the pool for that quarter and vintage
    (set to NaN; the long tables drop it). The guard catches the explosive readings of linear cells on raw designs;
    no Trinidad quarter has exceeded 12.3 in absolute value, so a true crash passes."""
    df = df.copy(); cols = [c for c in cols if c in df.columns]
    bad = df[cols].abs() > cfg.READING_GUARD; n = int(bad.values.sum())
    df[cols] = df[cols].mask(bad)
    if n: print(f"[reading guard {label}] {n} readings beyond |{cfg.READING_GUARD:g}| dropped ({int(bad.any(axis=1).sum())} cell-quarters)")
    return df
HEADLINE_EXCLUDE = set()     # kept for older cells; the rule now lives in pool_mask
run = dict(cfg.RUN)   # estimator switches, config section 5

# Parameter: iterations in GBT, DT, RF
_iter_ = cfg.ITER

# =========================================================
# Imports, paths, data loading and deflation
# =========================================================
# 1) the libraries (sklearn, torch + nowcast_lstm, statsmodels)
# 2) the project folders from `P` and this run's output folder
# 3) the panel and its metadata
# 4) nominal series deflated with the CPI (base year CPI_BASE_YEAR)
# Seasonal adjustment, growth rates and the panel filters follow in the filters cell.
# =========================================================

import os, json, time, warnings, re
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# -----------------------------
# Models / ML utilities
# -----------------------------
from sklearn import linear_model, tree
from sklearn.linear_model import (
    LinearRegression, RidgeCV, ElasticNet, Lasso, Lars
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Deep learning (LSTM nowcasting)
import torch
from nowcast_lstm.LSTM import LSTM

# Statsmodels seasonal adjustment
from statsmodels.tsa.seasonal import seasonal_decompose

# Reduce noisy warnings from some estimators / decomposition
warnings.filterwarnings("ignore", category=UserWarning)

# =========================================================
# Paths
# =========================================================
# Define local project directories (relative to current working directory)
PATH_RAW, PATH_DATA = P["raw"], P["data"]                       # from cfg.derive_paths
PATH_OUTPUT = os.path.join(P["output"], cfg.OUTPUT_SUBFOLDER)    # output/<date>_tto_prod or _ident
# Create folder if it does not exist
os.makedirs(PATH_OUTPUT, exist_ok=True)

# =========================================================
# Load data and metadata
# =========================================================
# Main modeling dataset (indexed by date)
data = pd.read_csv(
    os.path.join(PATH_DATA, data_file ),
    parse_dates=["date"],
    index_col="date"
)

test_end_date = data[target_variable].dropna().index.max()        # last published quarter
if cfg.TEST_END_ACTIVE:                                            # the identification run stops earlier
    test_end_date = min(test_end_date, pd.to_datetime(cfg.TEST_END_ACTIVE))

# Metadata describing each variable (frequency, nominal flag, seasonal-adjustment flag, publication lag, aggregation method)
metadata = pd.read_csv(os.path.join(PATH_DATA, meta_file))

# Key column names and CPI series ID used for deflation
meta_col = "ticket"     # variable identifier column in metadata
cpi = cfg.CPI_TICKER    # CPI series used to deflate nominal variables

# =========================================================
# Deflate nominal variables with the CPI (base year CPI_BASE_YEAR, config section 1)
# =========================================================
# Many series are nominal; deflating them puts everything in real terms.
base = data[cpi][data.index.year == cfg.CPI_BASE_YEAR].mean()

# Identify variables flagged as nominal in metadata
nominal = metadata.loc[metadata["nominal"] == 1][meta_col].values

# Deflate: Real_t = Nominal_t / CPI_t * CPI_base
for column in data.columns:
    if column in nominal:
        data[column] = data[column] / data[cpi] * base


## 3. Filters, redundancy rules on the levels, seasonal adjustment, growth rates

From deflated levels to the candidate panel: minimum observations, degenerate and duplicate series dropped on the levels, seasonal adjustment (X-13ARIMA-SEATS, classical decomposition as the fallback), growth rates, the missing-share / last-observation / publication-lag filters, and the panel-level data-quality guard. Writes `data_processed.csv` and `meta_processed.csv`.


In [ ]:
# [tto:filters]
# Filters: from deflated levels to the candidate panel of growth rates.
# Reads:  `data`, `metadata` (data cell); cfg sections 1, 2 and 7 (SA_METHOD).
# Writes: quarterly / monthly / weekly / daily ticket lists; `data` (seasonally adjusted growth rates from TRAIN_START,
#         filtered); `metadata` (restricted to the panel, months_lag as int); data_processed.csv, meta_processed.csv,
#         <date> 3_nwcst Tab panel_guard.csv.
# Order:  minimum observations -> redundancy rules A0/A on the levels -> seasonal adjustment -> growth rates ->
#         missing-share, last-observation and publication-lag filters -> panel-level data-quality guard.
# =========================================================
# Frequency groups and minimum observations
# =========================================================
quarterly = metadata.loc[metadata["freq"].str.lower().str.contains("q", na=False)][meta_col].values
monthly   = metadata.loc[metadata["freq"].str.lower().str.contains("m", na=False)][meta_col].values
weekly    = metadata.loc[metadata["freq"].str.lower().str.contains("w", na=False)][meta_col].values
daily     = metadata.loc[metadata["freq"].str.lower().str.contains("d", na=False)][meta_col].values

for column in list(data.columns):
    if len(data[column].dropna(how="all", axis=0)) < cfg.MIN_OBS:          # MIN_OBS observations
        data.drop(column, axis=1, inplace=True)

# =========================================================
# Redundancy rules computed on the LEVELS
#   A0 degenerate: constant over the last DEGENERATE_WINDOW observations or fewer than DEGENERATE_MIN_DISTINCT distinct values
#   A  exact duplicates across files: identical values on the overlap, the first ticket is kept
# The typed lists (B mechanical, C identities, D net balances, E by decision) are applied at the pre-selection cell.
# =========================================================
_degenerate = [c for c in data.columns if c != target_variable and (
    data[c].dropna().tail(cfg.DEGENERATE_WINDOW).nunique() <= 1 or data[c].dropna().nunique() < cfg.DEGENERATE_MIN_DISTINCT)]
data = data.drop(columns=_degenerate)
_duplicates = []
_live = [c for c in data.columns if c != target_variable]
for _i, _a in enumerate(_live):
    for _b in _live[_i + 1:]:
        if _b in _duplicates: continue
        _both = data[[_a, _b]].dropna()
        if len(_both) >= cfg.MIN_OBS and np.allclose(_both[_a], _both[_b], rtol=1e-6, atol=1e-9):
            _duplicates.append(_b)
data = data.drop(columns=_duplicates)
print(f"[redundancy A0/A] dropped {len(_degenerate)} degenerate and {len(_duplicates)} duplicate series on the levels")

# =========================================================
# Seasonal adjustment (only for non-SA variables)
# =========================================================
# Variables with sa==1 in metadata are already seasonally adjusted by the source.
# For all others, subtract only the seasonal component (keeps trend + residual/shocks).
non_sa = metadata.loc[metadata["sa"] == 0, meta_col].values

if cfg.SA_METHOD == "x13":
    # X-13ARIMA-SEATS on every non-SA series, target included. Falls back to the classical decomposition per series
    # when X-13 fails (short, monotonic or degenerate series). Both methods use the full sample for the factors.
    import os as _os, sys as _sys, shutil as _shutil
    from statsmodels.tsa.x13 import x13_arima_analysis
    # The US Census binary is looked up in the running environment (conda-forge package `x13as`), then on the PATH.
    # Without it every series takes the classical fallback below, so the run still completes.
    _x13_candidates = ([_os.path.join(_sys.prefix, "bin", n) for n in ("x13as_ascii", "x13as")]
                       + [_os.path.join(_sys.prefix, "Library", "bin", n) for n in ("x13as_ascii.exe", "x13as.exe")]
                       + [_shutil.which("x13as_ascii"), _shutil.which("x13as")])
    X13_BINARY = next((p for p in _x13_candidates if p and _os.path.exists(p)), None)
    if X13_BINARY is None:
        print("[SA x13] X-13ARIMA-SEATS binary not found (conda install -c conda-forge x13as); "
              "every series takes the classical decomposition in this run.")
    else:
        _os.environ.setdefault("X13PATH", _os.path.dirname(X13_BINARY))
    _n_x13 = _n_fallback = 0
    for column in list(data.columns):
        if column not in non_sa:
            continue
        series = data[column].dropna()
        if len(series) < 36:                       # X-13 needs >= 36 observations to estimate seasonality
            continue
        period = 4 if column in quarterly else 12
        try:
            freq_str = "Q" if period == 4 else "M"
            s_idx = series.copy(); s_idx.index = pd.PeriodIndex(s_idx.index, freq=freq_str)
            if X13_BINARY is None:
                raise RuntimeError("no X-13 binary")
            result = x13_arima_analysis(s_idx, x12path=X13_BINARY, prefer_x13=True, outlier=True, print_stdout=False)
            sa = result.seasadj.copy()
            if len(sa) == len(series):
                sa.index = series.index
            else:
                sa_idx = sa.index.to_timestamp()
                if period == 4: sa_idx = sa_idx + pd.DateOffset(months=2)
                sa.index = sa_idx
            data[column] = sa.reindex(data[column].index).combine_first(data[column]); _n_x13 += 1
        except Exception:
            try:
                decomp = seasonal_decompose(series, model="additive", extrapolate_trend="freq", period=period)
                data[column] = data[column] - decomp.seasonal; _n_fallback += 1
            except Exception:
                pass
    print(f"[SA x13] {_n_x13} series adjusted with X-13ARIMA-SEATS, {_n_fallback} with the classical fallback")
else:
    for column in list(data.columns):
        if column not in non_sa:
            continue
        series = data[column].dropna()
        if len(series) < 24:
            continue
        period = 4 if column in quarterly else 12
        try:
            decomp = seasonal_decompose(
                series,
                model="additive",
                extrapolate_trend="freq",
                period=period,
            )
            data[column] = data[column] - decomp.seasonal
        except Exception:
            pass  # leave as-is if decomposition fails

n_adjusted = sum(1 for c in data.columns if c in non_sa)
print(f"Seasonally adjusted {n_adjusted} non-SA series ({cfg.SA_METHOD}).")

# =========================================================
# Convert levels to growth rates (percent change)
# =========================================================
# Quarterly series: quarter-over-quarter growth rate
# All other series: default percent change at their observed frequency
for column in data.columns:
    if column in quarterly:
        # Get non-NaN values with their original index
        non_nan = data[column].dropna()
        
        # Calculate percent change on non-NaN values only
        pct_change_values = non_nan.pct_change(periods=1, fill_method=None) * 100
        
        # Create a new series filled with NaN, then update with calculated values
        result = pd.Series(np.nan, index=data.index)
        result.loc[pct_change_values.index] = pct_change_values
        
        data[column] = result
    else:
        # Store original NaN mask
        original_was_nan = data[column].isna()
        
        # For monthly/weekly/daily, calculate normally
        pct_change_values = data[column].pct_change(periods=1, fill_method=None) * 100
        
        # Set to NaN where previous value was NaN OR current original value was NaN
        prev_was_nan = data[column].shift(1).isna()
        pct_change_values[prev_was_nan | original_was_nan] = np.nan
        
        data[column] = pct_change_values

# =========================================================
# Training-window history rules (on growth rates, before the cut to TRAIN_START)
#   (i)  monthly and higher-frequency series: at most MAX_MISSING_TRAIN of the months up to the test start missing
#        (quarterly series are exempt from this one: a complete quarterly series is 2/3 "missing" on the monthly index)
#   (ii) every series: at least MIN_TRAIN_HISTORY_Q quarters of real observations inside [TRAIN_START, test start),
#        8 quarters for a quarterly series, 24 months otherwise. A series that starts inside or after the window would
#        otherwise be ranked, scored and estimated on its back-filled stretch. Each run applies the rule to its own window:
#        a series born in 2019-Q4 is out of the identification run (test from 2019-Q4) and in the production run (2023-Q2).
# =========================================================
_win = data[(data.index >= train_start_date) & (data.index < test_start_date)]
_short_history = []
for column in list(data.columns):
    if column == target_variable:
        continue
    if column not in quarterly:                                                            # (i)
        missing_pct = data[data.index <= test_start_date][column].isna().mean()
        if missing_pct > cfg.MAX_MISSING_TRAIN:   # 0.75
            data.drop(column, axis=1, inplace=True)
            continue
    _need = cfg.MIN_TRAIN_HISTORY_Q if column in quarterly else 3 * cfg.MIN_TRAIN_HISTORY_Q   # (ii)
    if _win[column].notna().sum() < _need:
        _short_history.append(column)
        data.drop(column, axis=1, inplace=True)
print(f"[history] dropped {len(_short_history)} series with fewer than {cfg.MIN_TRAIN_HISTORY_Q} quarters of real observations "
      f"between {train_start_date.date()} and {test_start_date.date()}: {_short_history}")

# =========================================================
# Final cleaning
# =========================================================
# Replace infinities created by pct_change with NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop the CPI from the predictor set: it deflates the nominal series (data cell) and is not a candidate itself.
# Active since 2026-11-09; the runs before that date kept its growth rate (monthly inflation) in the candidate panel.
data.drop(columns=[cpi], inplace=True, errors="ignore")

# Keep data from the training start onward and remove fully-missing rows
data = data[~(data.index < train_start_date)].reset_index(drop=False)
data = data.set_index("date").dropna(how="all", axis=0)

# The metadata may call the publication lag `publication_lag`; the pipeline uses `months_lag`
if "months_lag" not in metadata.columns and "publication_lag" in metadata.columns:
    metadata["months_lag"] = metadata["publication_lag"]

metadata["months_lag"] = (
    pd.to_numeric(metadata["months_lag"], errors="coerce")
      .fillna(1)
      .astype(int)
)

# Drop those columns
data = data.drop(columns=[col for col in data.columns if 'RGDP' in col and col != 'RGDP0000'])
for col in data.columns :
    last = data[col].dropna().index.max().year
    if last < pd.to_datetime(cfg.LAST_OBS_MIN).year :        # 2024
        data.drop(col, axis=1, inplace=True)
# Drop high-publication lag series
tickets_to_keep = metadata[metadata.months_lag <= cfg.MAX_PUB_LAG]['ticket'].tolist()   # MAX_PUB_LAG months
# Keep only these columns (that exist in data)
data = data[[ col for col in data.columns if col in tickets_to_keep or "RGDP" in col ]]

# =========================================================
# Data-quality guard as a PANEL rule: a series whose standardised growth rate, against its training-window
# distribution, exceeds MAX_ABS_Z anywhere leaves the candidate panel before the race (rates near zero, past-due
# loans, debt flows: level shifts that blow up the linear cells). The list goes to panel_guard.csv for the data team.
# =========================================================
_tr_mask = data.index < test_start_date
_pred_cols = [c for c in data.columns if c != target_variable]
_z_panel = (data[_pred_cols] - data.loc[_tr_mask, _pred_cols].mean()) / data.loc[_tr_mask, _pred_cols].std().replace(0, np.nan)
_worst_panel = _z_panel.abs().max()
_extreme = sorted(_worst_panel[_worst_panel > cfg.MAX_ABS_Z].index)
pd.DataFrame({"ticket": _extreme, "max_abs_z": _worst_panel[_extreme].round(1).values,
              "at": _z_panel[_extreme].abs().idxmax().dt.strftime("%Y-%m").values,
              "description": [metadata.set_index(meta_col)["description"].get(t, "") for t in _extreme]}) \
  .sort_values("max_abs_z", ascending=False).to_csv(os.path.join(PATH_OUTPUT, f"{date_str} 3_nwcst Tab panel_guard.csv"), index=False)
data = data.drop(columns=_extreme)
print(f"[panel guard] dropped {len(_extreme)} series with |z| > {cfg.MAX_ABS_Z} against the training window "
      f"(families: {pd.Series([t[:4] for t in _extreme]).value_counts().head(6).to_dict()}); panel now {data.shape[1] - 1} candidates")

data.to_csv(os.path.join(PATH_OUTPUT, "data_processed.csv"), index=True)
metadata = metadata[metadata.ticket.isin(data.columns)]
metadata.to_csv(os.path.join(PATH_OUTPUT, "meta_processed.csv"), index=True)
# =========================================================
# Reproducibility: print core library versions
# =========================================================
print("================== Library Versions =================")
import sklearn, numpy, pandas
print("scikit-learn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)


# Data dimensions (observations and variables)
print("Data dimensions:", data.shape)
print(f"Training dates: {train_start_date.strftime('%Y-%m-%d')} to {(test_start_date - pd.DateOffset(days=1)).strftime('%Y-%m-%d')}")
T1 = data[data.index < test_start_date].shape[0]
T2 = data[(data.index >= test_start_date) & (data.index <= test_end_date)].shape[0]
print(f"Training periods: {T1} ({T1/(T1+T2) :.2%})")
print(f"Testing dates: {test_start_date.strftime('%Y-%m-%d')} to {test_end_date.strftime('%Y-%m-%d')}")
print(f"Testing periods: {T2} ({T2/(T1+T2) :.2%})")


## 4. Model library

`nwcst_helpers.py`: the six screens, the data helpers and one fit / predict / oos triple per estimator.


In [ ]:
# [tto:helpers]
# Model library: nwcst_helpers.py beside this notebook (the six screens, the data helpers, fit_/predict_/oos_ per estimator).
# Reads:  CODE_DIR (setup cell).
# Writes: every public name of nwcst_helpers into the notebook namespace.
# joblib/loky workers re-import the module, so its folder must be importable -- hence the sys.path insert.
import sys, os
for _p in (os.getcwd(), CODE_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import nwcst_helpers
from nwcst_helpers import *   # estimate_factors, lars_variable_ranking, rank_*, gen_lagged_data, flatten_data,
                              # ffill_dataset, lagged_target, perform_tab, forecast_table, fit_/predict_/oos_ ...


## 5. Pre-selection: redundancy lists, six-screen race, groups, shock features

Four cells. **5.1** the quarterly panel and the typed redundancy lists (mechanical twins, small identity components, net balances, exclusions by decision). **5.2** the race: six screens (LARS, SIS, t-stat, SHAP, OCMT, iBMA) ranked once on the panel and again at every test quarter; every (screen, size) scored by expanding-window CV against the all-candidates PCA+OLS benchmark. **5.3** the groups: benchmark-beating sizes per screen (spread pick above the cap), consensus groups, rolling specs, frozen groups. **5.4** shock features (quarterly minimum and range per monthly variable), the data-quality guard, SHOK0000 and `estimation_parameters.json`.


### 5.1 Quarterly panel and redundancy rules


In [ ]:
# [tto:presel_panel]
# Pre-selection 1 of 4: the quarterly panel and the redundancy rules applied before any ranking.
# Reads:  `data`, `metadata` (filters cell); cfg sections 2 and 3; the level file for the identity shares.
# Writes: `pre_selected` (quarterly panel up to the last published quarter, redundant series dropped), first_obs, factors,
#         CORR_PRUNE / SHOCK_SHARE / SMALL_SHARE, _drop / _nets / _excluded_by_decision (logged in estimation_parameters.json).
# The four pre-selection cells: panel -> race -> groups -> shock features.
import warnings
warnings.filterwarnings('ignore')

factors = cfg.FACTORS   # 1: Bai-Ng IC1 consistently selects 1 factor given our sample size

# 1. Aggregate to quarterly (no qmin/qrange here — keeps benchmark fair)
try:
    agg_map = metadata.set_index('ticket')['agg_method'].to_dict()
    agg_funcs = {
        col: (agg_map[col] if agg_map[col] in ['mean', 'last', 'sum'] else 'mean')
        for col in data.columns if col in agg_map
    }
    pre_selected = data.resample('QS').agg(agg_funcs)
except:
    pre_selected = data.resample('QS').mean()

pre_selected.index = pd.to_datetime(pre_selected.index)
pre_selected = pre_selected.dropna(how='all', axis=1)
pre_selected = pre_selected.loc[:, pre_selected.isnull().mean() <= 0.75]
first_obs = pre_selected.apply(lambda s: s.first_valid_index())   # first real quarter per candidate, taken BEFORE the back-fill (used by the race CV)
pre_selected = pre_selected.ffill().bfill()
pre_selected = pre_selected[pre_selected.index <= data[target_variable].dropna().index.max()]   # up to the last published quarter

print(f"Pre-selected shape (quarterly): {pre_selected.shape}")

# =========================================================
# Redundancy rules B-E (typed lists in config section 2), applied to the quarterly panel BEFORE any ranking
# =========================================================
CORR_PRUNE  = cfg.CORR_PRUNE    # 0.8, 1b: max |corr| allowed with a variable already kept in a ranking
SHOCK_SHARE = cfg.SHOCK_SHARE   # 0.05: SHOK0000 fires when >= this share of the monthly qmin variables (floor 4) break -3 sigma
SMALL_SHARE = cfg.SMALL_SHARE   # 0.20, 1c: a component below this share of its aggregate is dropped

# 1a. mechanical derivations of other series (typed list B in config: 30-day NTL windows, yoy credit growth, debt ratios, import cover)
_mechanical = [c for c in pre_selected.columns if c in cfg.MECHANICAL]

# 1c. additive identities (typed list C in config, verified parts/aggregate = 1 on the October 2026 levels)
IDENTITIES = cfg.IDENTITIES
_levels = pd.read_csv(os.path.join(PATH_DATA, data_file), parse_dates=["date"], index_col="date")
_small = sorted({p for a, parts in IDENTITIES.items() if a in _levels
                   for p in parts if p in _levels and (_levels[p] / _levels[a]).mean() < SMALL_SHARE})
_nets = [c for c in pre_selected.columns if c in cfg.NET_BALANCES]                 # 1d: signed net balances, differences of gross flows in the panel
_excluded_by_decision = list(cfg.EXCLUDED_BY_DECISION)                          # 1e: typed list E in config, reasons logged
_drop = [c for c in dict.fromkeys(_mechanical + _small + _nets + _excluded_by_decision) if c in pre_selected.columns]
pre_selected = pre_selected.drop(columns=_drop)
print(f"[redundancy] dropped {len(_drop)} series: {sum(c in _mechanical for c in _drop)} mechanical twins, "
      f"{sum(c in _small for c in _drop)} small identity components {sorted(c for c in _drop if c in _small)}, "
      f"{sum(c in _nets for c in _drop)} net balances, "
      f"{sum(c in _excluded_by_decision for c in _drop)} by decision {_excluded_by_decision}")
print(f"[redundancy] panel now {pre_selected.shape[1] - 1} candidates")



### 5.2 The race


In [ ]:
# [tto:presel_race]
# Pre-selection 2 of 4: the six-screen race.
# Reads:  `pre_selected`, first_obs, factors, CORR_PRUNE (panel cell); cfg section 3; the rank_* screens (helpers cell).
# Writes: METHODS, rankings (static, per screen), rolling_selections {screen: {test date: ordered list}}, _test_dates,
#         benchmark (CV MSE of PCA+OLS on all candidates), `d` (CV MSE and ratio per screen and size), _cv_mse(),
#         ratio_curves, overlap, ov_mat, summary; two figures and the pre-selection workbook (4d).
def _prune(ranking, panel, thr=CORR_PRUNE):
    """1b. Walk down a ranking; keep a variable only if |corr| with every variable already kept is below thr."""
    cols = [v for v in ranking if v in panel.columns]
    C = panel[cols].corr().abs()
    kept = []
    for v in cols:
        if not kept or C.loc[v, kept].max() < thr:
            kept.append(v)
    return kept

# ---------------------------------------------
# 2. Rankings
# ---------------------------------------------
import numpy as np
METHODS          = ["lars", "sis", "tstat", "shap", "ocmt", "ibma"]
FIXED_SIZE       = min(cfg.FIXED_SIZE_MAX, pre_selected.shape[1] - 1)
MAX_ROLLING_VARS = cfg.MAX_ROLLING_VARS
gdp_lags_screen  = cfg.GDP_LAGS_SCREEN
hac_lags         = cfg.HAC_LAGS
RANK_ON_TRAIN_ONLY = cfg.RANK_ON_TRAIN_ONLY_ACTIVE   # prod True: static ranking on the training window; ident False: full panel
CV_ON_TRAIN_ONLY   = cfg.CV_ON_TRAIN_ONLY_ACTIVE     # prod True: every CV fold trains and validates before the test start; ident False

def _rank_all(panel, n_max):
    """One ordered list per method for the given quarterly panel (best first)."""
    out = {}
    out["lars"], _    = lars_variable_ranking(panel, target=target_variable,
                                               max_vars=min(n_max, panel.shape[1] - 1),
                                               test=None, factors=factors)
    out["sis"], _     = rank_sis(panel, target_variable)
    out["tstat"], _   = rank_tstat(panel, target_variable, gdp_lags=gdp_lags_screen, hac_lags=hac_lags)
    out["shap"], _    = rank_shap(panel, target_variable)
    out["ocmt"], _, _ = rank_ocmt(panel, target_variable, gdp_lags=gdp_lags_screen, hac_lags=hac_lags)
    out["ibma"], _    = rank_ibma(panel, target_variable)
    return {m: _prune(list(v), panel)[:n_max] for m, v in out.items()}   # 1b: prune static AND rolling rankings

# 2a. Rolling: at each test quarter, rank with data strictly BEFORE that quarter (per method)
_test_dates = [d for d in pre_selected.index
               if d >= pd.Timestamp(test_start_date) and d <= pre_selected.index.max()]
print(f"[Rolling] test dates: {len(_test_dates)} ({_test_dates[0].date()} -> {_test_dates[-1].date()})")

rolling_selections = {m: {} for m in METHODS}       # {method: {Timestamp: [ordered vars]}}
_t_roll = time.time()
for _t in _test_dates:
    _train_slice = pre_selected[pre_selected.index < _t]
    if len(_train_slice) < 16:
        for m in METHODS:
            rolling_selections[m][_t] = []
        continue
    _r = _rank_all(_train_slice, MAX_ROLLING_VARS)
    for m in METHODS:
        rolling_selections[m][_t] = _r[m]
    print(f"  [Rolling] {_t.date()}: " + ", ".join(f"{m}={len(_r[m])}" for m in METHODS), flush=True)
print(f"[Rolling] done in {time.time() - _t_roll:.0f}s")

# 2b. Static ranking (drives the size selection through the CV below)
_rank_panel = pre_selected[pre_selected.index < test_start_date] if RANK_ON_TRAIN_ONLY else pre_selected
rankings = _rank_all(_rank_panel, FIXED_SIZE)
for m in METHODS:
    print(f"{m.upper():6s} ranked {len(rankings[m]):3d} variables; top-5: {rankings[m][:5]}")

# ---------------------------------------------
# 3. Time-series CV evaluation (expanding window, no leakage): same folds and same benchmark for every method
# ---------------------------------------------
n         = len(pre_selected[pre_selected.index < test_start_date]) if CV_ON_TRAIN_ONLY else len(pre_selected)   # quarters a fold may use
min_train, step, eval_size = cfg.CV_MIN_TRAIN, cfg.CV_STEP, cfg.CV_EVAL_SIZE   # 16, 2, 4

fold_starts = list(range(min_train, n - eval_size, step))
print(f"Number of CV folds: {len(fold_starts)}" + (f" (all validation windows before {test_start_date.date()})" if CV_ON_TRAIN_ONLY else " (full panel)"))

CV_MIN_REAL_OBS = cfg.CV_MIN_REAL_OBS   # 8: no back-filled series inside a fold's design
CV_CLIP_Z       = cfg.CV_CLIP_Z         # 5: clip standardised designs; None = off

def _cv_mse(cols):
    """Average fold MSE of PCA(factors)+OLS on `cols` (None = all variables, i.e. the benchmark).
    Per fold, a candidate is admitted only with >= CV_MIN_REAL_OBS real quarters inside the training window
    (its back-filled stretch never enters a design); a group with no admitted candidate forecasts the training mean.
    Standardised designs are clipped at +/- CV_CLIP_Z so one outlying quarter cannot drive the factor."""
    mses = []
    for fs in fold_starts:
        fold_train = pre_selected.iloc[:fs]
        fold_val   = pre_selected.iloc[fs:fs + eval_size]
        y_tr = fold_train[target_variable].dropna()
        y_vl = fold_val[target_variable].dropna()
        if len(y_tr) < 5 or len(y_vl) < 1:
            continue
        candidates = pre_selected.columns.drop(target_variable) if cols is None else cols
        use = [c for c in candidates if (y_tr.index >= first_obs[c]).sum() >= CV_MIN_REAL_OBS]
        if not use:
            mses.append(np.mean((y_vl.values - y_tr.mean()) ** 2))
            continue
        X_tr = fold_train[use].loc[y_tr.index]
        X_vl = fold_val[use].loc[y_vl.index]
        sc   = StandardScaler()
        Z_tr = sc.fit_transform(X_tr)
        Z_vl = sc.transform(X_vl)
        if CV_CLIP_Z:
            Z_tr = np.clip(Z_tr, -CV_CLIP_Z, CV_CLIP_Z)
            Z_vl = np.clip(Z_vl, -CV_CLIP_Z, CV_CLIP_Z)
        ncomp = min(factors, X_tr.shape[1], len(y_tr) - 1)
        pca   = PCA(n_components=ncomp).fit(Z_tr)
        m     = LinearRegression().fit(pca.transform(Z_tr), y_tr)
        mses.append(np.mean((y_vl.values - m.predict(pca.transform(Z_vl))) ** 2))
    return np.mean(mses)

benchmark = _cv_mse(None)
print(f"Benchmark CV MSE: {benchmark:.4f}")

results = []
for m in METHODS:
    for size in range(5, min(FIXED_SIZE, len(rankings[m])) + 1):
        top_k     = rankings[m][:size]
        available = [v for v in top_k if v in pre_selected.columns]
        if len(available) < 2:
            continue
        avg_mse = _cv_mse(available)
        results.append({'method': m, 'size': size, 'selected_vars': ' '.join(top_k),
                        'rmse': avg_mse, 'ratio': avg_mse / benchmark})
d = pd.DataFrame(results).reset_index(drop=True)
print("Sizes beating the benchmark, by method:")
print(d[d['ratio'] < 1].groupby('method')['size'].count().reindex(METHODS).fillna(0).astype(int).to_string())

# ---------------------------------------------
# 4a. Ratio curves, one line per method
# ---------------------------------------------
ratio_curves = d.pivot(index='size', columns='method', values='ratio').reindex(columns=METHODS)
styles = {"lars": ("black", "-"), "sis": ("gray", "--"), "tstat": ("black", "-."),
          "shap": ("gray", ":"), "ocmt": ("steelblue", "-"), "ibma": ("firebrick", "--")}
fig, ax = plt.subplots(figsize=(8, 4))
for m in METHODS:
    color, ls = styles[m]
    ratio_curves.loc[ratio_curves.index > 10, m].plot(ax=ax, color=color, linestyle=ls, label=m.upper())
ax.axhline(y=1, color='black', linestyle=':', linewidth=1)
ax.set_ylabel("Ratio of CV MSE (selection / all-variables benchmark)")
ax.set_xlabel("Number of variables")
ax.legend(frameon=False, ncol=3)
plt.tight_layout()
plt.savefig(f"{PATH_OUTPUT}/{date_str} 3_nwcst Fig preselection_ratio.png", dpi=300, bbox_inches='tight')
plt.show()

# ---------------------------------------------
# 4b. Overlap with the incumbent (LARS) at every size + full matrix at k=60
# ---------------------------------------------
overlap = []
for size in range(5, FIXED_SIZE + 1):
    row = {"size": size}
    for m in METHODS[1:]:
        sa, sb = set(rankings["lars"][:size]), set(rankings[m][:size])
        row[f"lars-{m}"] = len(sa & sb) / size
    overlap.append(row)
overlap = pd.DataFrame(overlap).set_index("size")
fig, ax = plt.subplots(figsize=(8, 4))
for col, m in zip(overlap.columns, METHODS[1:]):
    color, ls = styles[m]
    overlap[col].plot(ax=ax, color=color, linestyle=ls, label=col.upper())
ax.set_ylabel("Share of common variables vs LARS"); ax.set_xlabel("Number of variables"); ax.set_ylim(0, 1)
ax.legend(frameon=False, ncol=3)
plt.tight_layout()
plt.savefig(f"{PATH_OUTPUT}/{date_str} 3_nwcst Fig preselection_overlap.png", dpi=300, bbox_inches='tight')
plt.show()

k_ref = min(cfg.K_REF, FIXED_SIZE)
ov_mat = pd.DataFrame(index=METHODS, columns=METHODS, dtype=float)
for a in METHODS:
    for b in METHODS:
        ov_mat.loc[a, b] = len(set(rankings[a][:k_ref]) & set(rankings[b][:k_ref])) / k_ref
print(f"Overlap matrix at k={k_ref} (share of common variables):")
print(ov_mat.round(2))

# ---------------------------------------------
# 4c. Head-to-head summary
# ---------------------------------------------
_rows = []
for m in METHODS:
    g = d[d['method'] == m]
    if g.empty:
        continue
    _rows.append({"method": m,
                  "specs_below_threshold": int((g['ratio'] < 1).sum()),
                  "best_ratio": g['ratio'].min(),
                  "best_size": int(g.loc[g['ratio'].idxmin(), 'size']),
                  "median_ratio": g['ratio'].median()})
summary = pd.DataFrame(_rows).set_index("method")
print("Head-to-head summary (lower ratio = better):")
print(summary)

# ---------------------------------------------
# 4d. Export (ratio curves, MSE levels, overlap, summary, run context)
# ---------------------------------------------
with pd.ExcelWriter(f"{PATH_OUTPUT}/{date_str} 3_nwcst preselection_ratio.xlsx", engine="openpyxl") as writer:
    ratio_curves.round(6).to_excel(writer, sheet_name="ratio_curves")
    d.pivot(index="size", columns="method", values="rmse").reindex(columns=METHODS).round(6).to_excel(writer, sheet_name="mse_levels")
    overlap.round(4).to_excel(writer, sheet_name="overlap_vs_lars")
    ov_mat.round(4).to_excel(writer, sheet_name=f"overlap_k{k_ref}")
    summary.to_excel(writer, sheet_name="summary")
    pd.DataFrame({"item": ["benchmark_cv_mse", "factors_pca", "train_start", "test_start", "test_end",
                           "n_variables_panel", "rank_on_train_only", "cv_on_train_only", "cv_min_real_obs", "cv_clip_z", "n_test_dates_rolling"],
                  "value": [benchmark, factors, str(train_start_date), str(test_start_date), str(test_end_date),
                            pre_selected.shape[1] - 1, RANK_ON_TRAIN_ONLY, CV_ON_TRAIN_ONLY, CV_MIN_REAL_OBS, CV_CLIP_Z, len(_test_dates)]}
                 ).to_excel(writer, sheet_name="run_info", index=False)
print(f"Preselection tables exported to {PATH_OUTPUT}/{date_str} 3_nwcst preselection_ratio.xlsx")



### 5.3 From rankings to groups


In [ ]:
# [tto:presel_groups]
# Pre-selection 3 of 4: from the race to the list of groups.
# Reads:  `d`, benchmark, rankings, rolling_selections, _test_dates, _cv_mse() (race cell); cfg sections 3 and 4.
# Writes: spec_list (one row per group: method, K, selected_vars, rmse, ratio, rolling_enabled, augment),
#         chosen_methods, max_specs_per_method, common_vars, top_vars.
# Rows:   per screen the sizes beating the benchmark (spread pick above the cap) -> consensus groups (common, shared)
#         -> rolling specs re-pointed at their screen's per-date selection -> frozen groups from config.
# ---------------------------------------------
# 5. spec_list: per method, the sizes that beat the benchmark (ratio < 1); if no method beats it, the 15 best
#    sizes overall by CV MSE; optional cap per method (spread pick).
# ---------------------------------------------
chosen_methods       = cfg.CHOSEN_METHODS         # None = every method with at least one benchmark-beating group
max_specs_per_method = cfg.MAX_SPECS_PER_METHOD   # a method with MORE beating groups than this is thinned by _spread_pick

def _spread_pick(sl, n):
    """Option 2: best group (lowest CV MSE) in each of n equal-width bins of K; empty bins filled with the best remaining."""
    if len(sl) <= n:
        return sl
    edges = np.linspace(sl['size'].min(), sl['size'].max() + 1e-9, n + 1)
    picks = [b.loc[b['rmse'].idxmin()] for lo, hi in zip(edges[:-1], edges[1:])
             for b in [sl[(sl['size'] >= lo) & (sl['size'] < hi)]] if len(b)]
    p = pd.DataFrame(picks)
    if len(p) < n:
        p = pd.concat([p, sl[~sl.index.isin(p.index)].nsmallest(n - len(p), 'rmse')])
    return p.sort_values('size')

_parts = []
for m in (chosen_methods or METHODS):
    dm = d[d['method'] == m]
    sl = dm[dm['ratio'] < 1].copy()
    if len(sl) == 0:
        print(f"[skip] {m}: no size beats the benchmark")
        continue
    if max_specs_per_method is not None:
        sl = _spread_pick(sl, max_specs_per_method)   # thins only methods with more groups than the cap
    _parts.append(sl)
if not _parts:   # nothing beats the benchmark -> the 15 best sizes overall by CV MSE
    print("[soft-fallback] no method beats the benchmark; using the 15 best sizes overall")
    _parts.append(d.nsmallest(15, 'rmse').copy())
spec_list = pd.concat(_parts, axis=0).reset_index(drop=True)
top_vars = spec_list.loc[spec_list['rmse'].idxmin(), 'selected_vars'].split(' ')
print(f"Total specifications selected: {len(spec_list)} "
      f"({spec_list['method'].value_counts().reindex(METHODS).fillna(0).astype(int).to_dict()})")

# ---- consensus groups across the benchmark-beating methods (static rows) ----
_beating = [m for m in METHODS if (d[d['method'] == m]['ratio'] < 1).any()]
_sets = {m: set(d[(d['method'] == m) & (d['ratio'] < 1)].sort_values('size').iloc[-1]['selected_vars'].split()) for m in _beating}
_count = pd.Series([v for s_ in _sets.values() for v in s_]).value_counts()
common_vars = []
KEEP_COMMON = cfg.KEEP_COMMON   # the common group is a fixed row like the economist sets (it is the watchlist reference);
                     # its CV ratio is reported, not applied. The 'shared' group stays under the ratio rule.
for _name, _min in [("common", len(_beating)), ("shared", max(len(_beating) - 1, 2))]:   # all methods / all but one
    _vars = sorted(_count[_count >= _min].index,                       # ties in the summed rank break by name, so the
                   key=lambda v: (sum(rankings[m].index(v) + 1 for m in _beating if v in rankings[m]), v))   # order is the same in every process
    if len(_vars) >= 3 and (_name == "common" or len(_vars) > len(common_vars)):
        _mse = _cv_mse(_vars)
        _fixed = (_name == "common" and KEEP_COMMON)
        if _mse / benchmark < 1 or _fixed:
            spec_list = pd.concat([spec_list, pd.DataFrame([{'method': _name, 'size': len(_vars), 'selected_vars': ' '.join(_vars),
                                                             'rmse': _mse, 'ratio': _mse / benchmark}])], ignore_index=True)
            _how = "kept as a fixed row (ratio reported, not applied)" if (_fixed and _mse / benchmark >= 1) else "kept"
            print(f"[{_name}] {len(_vars)} variables in >= {_min} of {len(_beating)} beating methods: {_vars}  CV ratio {_mse / benchmark:.2f} -> {_how}")
        else:
            print(f"[{_name}] {len(_vars)} variables in >= {_min} of {len(_beating)} beating methods: CV ratio {_mse / benchmark:.2f} -> NOT kept (does not beat the benchmark)")
    if _name == "common":
        common_vars = _vars

# ---- rolling specs: each references the per-date selection of ITS method, sliced at K ----
spec_list = spec_list.rename(columns={'size': 'K'})
_roll = spec_list['method'].isin(rolling_selections.keys())
spec_list.loc[_roll, 'selected_vars'] = spec_list[_roll].apply(
    lambda r: ' '.join(rolling_selections[r['method']][_test_dates[-1]][:int(r['K'])]), axis=1)
spec_list['rolling_enabled'] = _roll.astype(int)
print(f"[specs] {len(spec_list)} (method, K) specs (rolling enabled)")

# ---- frozen groups (config section 4): static rows outside the race, five designs like any group ----
if getattr(cfg, "FROZEN_GROUPS", None):
    _frz_rows, _frz_missing = [], {}
    for g in cfg.FROZEN_GROUPS:
        _have = [t for t in g["selected_vars"].split() if t in pre_selected.columns]
        _frz_missing[g["spec"]] = [t for t in g["selected_vars"].split() if t not in pre_selected.columns]
        _frz_rows.append({"method": g["method"], "K": int(g["K"]), "selected_vars": " ".join(_have), "rmse": np.nan, "ratio": np.nan, "rolling_enabled": 0,
                          "augment": int(bool(g.get("augment", False)))})   # 0 = race design (base tickets only), 1 = static design with N/R companions
    spec_list = pd.concat([spec_list, pd.DataFrame(_frz_rows)], ignore_index=True)
    print(f"[frozen] {len(_frz_rows)} frozen groups appended (static); tickets missing from this panel: {_frz_missing}")
print(f"Best-CV spec: {spec_list.loc[spec_list['rmse'].idxmin(), 'method'].upper()} with {len(top_vars)} variables")



### 5.4 Shock features and data-quality guard


In [ ]:
# [tto:presel_shock]
# Pre-selection 4 of 4: within-quarter shock features, the data-quality guard and the shock indicator.
# Reads:  spec_list, `data`, `metadata`, monthly (earlier cells); SHOCK_SHARE; cfg MAX_ABS_Z.
# Writes: `data` with <ticket>N (quarterly minimum) and <ticket>R (quarterly range) per selected monthly variable and
#         SHOK0000; `metadata` rows for them; spec_list.selected_vars augmented (augment = 1 rows); broken_vars;
#         pre_selected_columns_monthly (the fixed panel of the breadth indicator); estimation_parameters.json;
#         <date> 3_nwcst Tab data_quality.csv.
# ---- shock features for the monthly variables of every group ----
# Naming convention follows existing tickets (alphanumeric, no underscores):
#   <8-char-base>N  -> quarterly MINimum  (e.g. MLOAB004 -> MLOAB004N)
#   <8-char-base>R  -> quarterly Range    (e.g. MLOAB004 -> MLOAB004R)

all_base_vars = set()
for _, row in spec_list.iterrows():
    all_base_vars.update(row['selected_vars'].split())

base_monthly_vars = sorted(v for v in all_base_vars if v in monthly)   # sorted: all_base_vars is a set, whose order is the
                                                                       # process's hash order; the N/R companion columns inherit
                                                                       # it, and trees and the LSTM read features by position
                                                                       # (otherwise their cells differ from one process to the next)

# ---- data-quality guard on the selected monthly variables ----
# A monthly series whose standardised growth, against its own training-window distribution, exceeds
# MAX_ABS_Z anywhere is treated as broken (level shift, reclassification, series near zero). It stays in
# the candidate panel (penalised estimators and trees cope with outliers) but is excluded from the
# within-quarter shock features and from the breadth indicator, and listed for the data team.
MAX_ABS_Z = cfg.MAX_ABS_Z   # 20
_tr = data.index < test_start_date
_z_full = (data[base_monthly_vars] - data[base_monthly_vars].loc[_tr].mean()) \
          / data[base_monthly_vars].loc[_tr].std().replace(0, np.nan)
_worst = _z_full.abs().max()
broken_vars = sorted(_worst[_worst > MAX_ABS_Z].index)
_desc = metadata.set_index("ticket")["description"]
pd.DataFrame({"ticket": _worst.index, "max_abs_z": _worst.values.round(1),
              "at": _z_full.abs().idxmax().dt.strftime("%Y-%m").values,
              "description": [_desc.get(t, "") for t in _worst.index],
              "excluded_from_shock_features": _worst.values > MAX_ABS_Z}) \
  .sort_values("max_abs_z", ascending=False) \
  .to_csv(os.path.join(PATH_OUTPUT, f"{date_str} 3_nwcst Tab data_quality.csv"), index=False)
print(f"[data quality] {len(broken_vars)} series beyond |z| > {MAX_ABS_Z} excluded from shock features "
      f"and the breadth count: {broken_vars}")
base_monthly_vars = [v for v in base_monthly_vars if v not in broken_vars]
print(f"Computing shock features for {len(base_monthly_vars)} LARS-selected monthly variables")

if base_monthly_vars:
    md = data[base_monthly_vars]

    qmin_full = md.resample('QS').min().reindex(data.index, method='ffill')
    qmin_full.columns = [c + 'N' for c in qmin_full.columns]

    qrange_full = (md.resample('QS').max() - md.resample('QS').min()).reindex(data.index, method='ffill')
    qrange_full.columns = [c + 'R' for c in qrange_full.columns]

    data = pd.concat([data, qmin_full, qrange_full], axis=1)

    # Synthesize metadata rows so gen_lagged_data() knows publication lags
    aug_cols = list(qmin_full.columns) + list(qrange_full.columns)
    aug_meta_rows = []
    for col in aug_cols:
        parent = col[:-1]   # strip final 'N' or 'R'
        parent_row = metadata[metadata['ticket'] == parent]
        if len(parent_row) > 0:
            new_row = parent_row.iloc[0].copy()
            new_row['ticket'] = col
            suffix_label = 'quarterly min' if col.endswith('N') else 'quarterly range'
            new_row['description'] = str(new_row.get('description', col)) + f' ({suffix_label})'
            aug_meta_rows.append(new_row)
    if aug_meta_rows:
        metadata = pd.concat([metadata, pd.DataFrame(aug_meta_rows)], ignore_index=True)

    # Expand each spec's selected_vars: base + baseN + baseR
    def _augment_spec_vars(vars_str):
        out = []
        for v in vars_str.split():
            out.append(v)
            if v in monthly:
                out.append(v + 'N')
                out.append(v + 'R')
        return ' '.join(out)

    # frozen groups with augment=0 keep the base tickets only, the design the rolling race cells use
    # (the loop takes the top-K base tickets per quarter, without N/R companions or SHOK0000); augment=1 = static design.
    _aug = spec_list['augment'].fillna(1).astype(bool) if 'augment' in spec_list.columns else pd.Series(True, index=spec_list.index)
    spec_list.loc[_aug, 'selected_vars'] = spec_list.loc[_aug, 'selected_vars'].apply(_augment_spec_vars)
    top_vars = spec_list.loc[spec_list['rmse'].idxmin(), 'selected_vars'].split(' ')

    print(f"After augmentation: best spec has {len(top_vars)} features")

    # ---- SHOK0000: binary shock indicator, share-based threshold ----
    # Shock flag: at least SHOCK_SHARE of the qmin variables (floor 4) are below -3 sigma of their
    # training-period distribution at the same time. A share scales with the panel; a fixed count does not.
    _train_mask = qmin_full.index < test_start_date
    _mu  = qmin_full.loc[_train_mask].mean()
    _sig = qmin_full.loc[_train_mask].std().replace(0, float("nan"))
    _z   = (qmin_full - _mu) / _sig
    _n_broken = (_z < -3).sum(axis=1).fillna(0).astype(int)
    _shock_thr = max(4, int(np.ceil(SHOCK_SHARE * qmin_full.shape[1])))
    _shock_col = (_n_broken >= _shock_thr).astype(int)
    print(f"  [is_shock] threshold {_shock_thr} of {qmin_full.shape[1]} monthly variables")
    _shock_col.name = 'SHOK0000'
    data['SHOK0000'] = _shock_col.reindex(data.index).fillna(0).astype(int)
    print(f"  [is_shock] flagged {int(_shock_col.sum())} rows across {_shock_col.groupby(_shock_col.index.to_period('Q')).max().sum()} quarters")

    # Synthetic metadata row so gen_lagged_data knows publication lag (treat as daily: 0 months lag)
    _shock_meta = metadata.iloc[0].copy()
    _shock_meta['ticket'] = 'SHOK0000'
    _shock_meta['series'] = 'shock_indicator'
    _shock_meta['description'] = 'binary shock indicator'
    _shock_meta['freq'] = 'daily'
    _shock_meta['months_lag'] = 0
    _shock_meta['agg_method'] = 'max'
    metadata = pd.concat([metadata, pd.DataFrame([_shock_meta])], ignore_index=True)

    # Add SHOK0000 to every spec's feature list
    spec_list.loc[_aug, 'selected_vars'] = spec_list.loc[_aug, 'selected_vars'].apply(lambda s: s + ' SHOK0000')   # not for base-only frozen groups
    top_vars = spec_list.loc[spec_list['rmse'].idxmin(), 'selected_vars'].split(' ')
    print(f"SHOK0000 added. Total shock-quarters flagged: {int(data['SHOK0000'].sum())}")

pre_selected_columns_monthly = [c for c in pre_selected.columns if c in monthly]   # fixed panel for the breadth indicator (post cell)
del pre_selected

# Save estimation parameters
parameters = {
    "train_start": train_start_date.strftime("%Y-%m-%d"),
    "test_start": test_start_date.strftime("%Y-%m-%d"),
    "test_end": test_end_date.strftime("%Y-%m-%d"),
    "lars_factors": factors,
    "corr_prune": CORR_PRUNE,
    "small_share": SMALL_SHARE,
    "shock_share": SHOCK_SHARE,
    "max_abs_z": MAX_ABS_Z,
    "excluded_from_shock_features": broken_vars,
    "excluded_by_decision": _excluded_by_decision,
    "net_balances_dropped": _nets,
    "n_dropped_redundant": len(_drop),
    "min_train_history_q": cfg.MIN_TRAIN_HISTORY_Q,
    "dropped_short_history": _short_history,
    "penalty_cv": "ElasticNetCV(l1_ratio 0.25/0.5/0.75, 50 alphas) + LassoCV(50 alphas), TimeSeriesSplit(4)",
    "consensus_groups": {"common": common_vars},
    "preselection_methods": METHODS,
    "preselection_chosen": chosen_methods,
    "max_specs_per_method": max_specs_per_method,
    "rank_on_train_only": RANK_ON_TRAIN_ONLY,
    "cv_on_train_only": CV_ON_TRAIN_ONLY,
    "cv_min_real_obs": CV_MIN_REAL_OBS,
    "cv_clip_z": CV_CLIP_Z,
    "screen_gdp_lags": gdp_lags_screen,
    "screen_hac_lags": hac_lags,
    "shap_model": "GBR(n_estimators=300, max_depth=2, lr=0.05, seed=0)",
    "ocmt_params": "alpha=0.05, delta=1, delta_star=1.5",
    "ibma_params": "w=10, pip_drop=0.05, BIC weights",
    "model_iterations": int(_iter_),
    **run
}
with open(os.path.join(PATH_OUTPUT, "estimation_parameters.json"), "w") as f:
    json.dump(parameters, f, indent=2)

print("Distribution of CV MSE in the selected specs, by method:")
print(spec_list.groupby('method')['rmse'].describe())
metadata[metadata['ticket'].isin(top_vars)]


## 6. Specifications, estimator map, economist groups

Each group becomes five designs (base, flats, p5, p3, p5l2); each design names the estimators that run on it; the economist groups are appended.


In [ ]:
# [tto:specs]
# Specifications: every group x five designs, the estimator map per design, the economist groups.
# Reads:  spec_list (pre-selection cells); cfg section 4 (HUMAN_GROUPS).
# Writes: spec_list with one row per (group, design): spec name, gdp_lags, PCA, data_flat, estimators.
# Designs: base (three-month flattening), flats (four), p5 and p3 (factor designs), p5l2 (p5 with two GDP lags).
import warnings
warnings.filterwarnings('ignore')
# --- Naming: <method><K>, e.g. lars43, shap28, ocmt46; the single best CV spec is "bench<K>" ---
spec_list = spec_list.copy().reset_index(drop=True)

spec_list["spec"] = spec_list["method"] + spec_list["K"].astype(int).astype(str)
bench_idx = spec_list["rmse"].idxmin()
spec_list.loc[bench_idx, "spec"] = "bench" + str(int(spec_list.loc[bench_idx, "K"]))

# --- Base settings (data_flat=3 only) ---
base = spec_list.assign(gdp_lags=0, PCA=0, data_flat=3)

variants = [base]

# --- Flats branch (data_flat=4 only, no PCA/lag variants); suffix so the method stays visible ---
flats = base.assign(data_flat=4)
flats["spec"] = flats["spec"] + "flats"
variants.append(flats)

# --- PCA variants ONLY on base (data_flat=3) ---
p5 = base.assign(PCA=5)
p5["spec"] = p5["spec"] + "p5"
variants.append(p5)

p3 = base.assign(PCA=3)
p3["spec"] = p3["spec"] + "p3"
variants.append(p3)

# --- Lags variant ONLY for p5 specs on base (data_flat=3) ---
p5_l2 = p5.copy()
p5_l2["gdp_lags"] = 2
p5_l2["spec"] = p5_l2["spec"] + "l2"
variants.append(p5_l2)

# --- Final spec_list ---
spec_list = pd.concat(variants, axis=0).reset_index(drop=True)
spec_list = spec_list[(spec_list["PCA"] == 0) | (spec_list["PCA"] < spec_list["K"])].reset_index(drop=True)   # PCA needs fewer factors than variables

# --- Estimator map per design ---
#   linear models (ridge, elastic net, LASSO, OLS if enabled): every design; on the raw designs (over a hundred
#       regressors on about thirty quarters) their readings can explode, which the reading guard (data cell) catches;
#   trees: every design; the single tree on factor designs recalls the pandemic leaf in shock quarters, which the
#       regime-aware weights silence in ordinary quarters and which carries pandemic memory into the shock consensus;
#   LSTM: only where the design differs for it -- it ignores flattening (flats == base) and GDP lags (l2 == p5).
# The worker runs only the estimators listed here; an empty/missing value means "all enabled".
LINEAR_EST, TREE_EST, LSTM_EST = ["ols", "olsr", "enet", "lasso"], ["dt", "dt4", "rf", "gbt"], ["lstm"]
# Decide by the row's design columns, not by the end of its name (a group of five variables is 'shap5', not a
# five-factor design).
def _estimators_for(row):
    if row["PCA"] > 0 and row["gdp_lags"] > 0:  return LINEAR_EST + TREE_EST              # p5l2
    if row["PCA"] > 0:                          return LINEAR_EST + TREE_EST + LSTM_EST   # p3, p5, economist sets
    if row["data_flat"] == 4:                   return LINEAR_EST + TREE_EST              # flats
    return LINEAR_EST + TREE_EST + LSTM_EST        # base: LSTM reads the raw monthly sequences here (not the stacked table)
spec_list["estimators"] = spec_list.apply(lambda r: " ".join(_estimators_for(r)), axis=1)
_fits_all = len(spec_list) * 7
_fits_map = int(spec_list["estimators"].str.split().apply(lambda l: len([e for e in l if e != "ols"])).sum())
print(f"estimator map: {_fits_map} estimator x spec cells instead of {_fits_all} ({_fits_map / _fits_all:.0%})")
assert spec_list["spec"].is_unique, "duplicate spec names"

print(f"Total iterations: {len(spec_list)}")
print(spec_list["method"].value_counts().to_dict())
spec_list

# =========================================================
# Economist groups (config section 4): human1 "energy", human2 "activity".
# Design: PCA 3, two GDP lags, three-month flattening, all estimators.
# =========================================================
N = [dict({k: v for k, v in _g.items() if k != "label"}, size=len(_g["selected_vars"].split())) for _g in cfg.HUMAN_GROUPS]

for _h in N :
    spec_list = pd.concat([spec_list, pd.DataFrame([_h])], axis=0, ignore_index=True)

print("Human-spec tickets missing from the processed data:",
      {h['spec']: [t for t in h['selected_vars'].split() if t not in data.columns] for h in N})
spec_list


## 7. Training loop (vintages, leave-one-out refit, out-of-sample readings) -- the engine is `nwcst_engine.py`

Two cells. **7.1** the options, the queue and the worker: one spec at every test quarter and publication lag, the leave-one-out refit of the shock quarter, the out-of-sample readings. **7.2** the parallel run and the saves. The seed runner executes 7.1 from this notebook and replaces 7.2.


### 7.1 The loop

One call of `nwcst_engine.run_one_spec` per specification, in parallel. The engine (six stages: frame, train a quarter, read it at the nine lags, leave-one-out refit of the shock quarter, tables, out-of-sample readings) and its estimator registry live in `nwcst_engine.py`; the seed runner imports the same function.


In [ ]:
# [tto:loop]
# Training loop: one call of nwcst_engine.run_one_spec per spec_list row, in parallel over rows (joblib/loky), the
# tables saved every SAVE_BATCH_SIZE specs and at the end. A spec already in the performance table on disk is skipped
# (resume); delete the tables in PATH_OUTPUT to start from spec 1.
# Reads:  data, metadata, spec_list, rolling_selections, run, lags, the sample dates (earlier cells); cfg section 5
#         (through nwcst_engine.make_opts) and MAX_WORKERS / SAVE_BATCH_SIZE / ITER_LSTM / EPOCH_LSTM.
# Writes: <date> 3_nwcst Tab perf / fcst / oos / shock_residuals .xlsx in PATH_OUTPUT; performance_table,
#         result_table, outofsample_table in the namespace.
# The seed runner (run_tto_seed.py) imports the same engine and replaces this cell with its own checkpointed loop.
import os, timeit
import pandas as pd
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from nwcst_engine import run_one_spec, make_context, make_opts, make_rows

for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = "1"                       # one BLAS thread per worker; set before heavy numeric work
torch.set_num_threads(2)
start = timeit.default_timer()

# 1) existing tables: a restarted run resumes after the specs already there
try:
    performance_table = pd.read_excel(f"{PATH_OUTPUT}/{performance_file}")
    result_table      = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", index_col="date")
    outofsample_table = pd.read_excel(f"{PATH_OUTPUT}/{outofsample_file}", index_col="date")
    print("Existing tables loaded.")
except Exception:
    performance_table, result_table, outofsample_table = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
done_specs = set(performance_table["spec"].unique()) if "spec" in performance_table.columns else set()

# 2) what the engine needs
ctx  = make_context(data, metadata, target_variable, lags, run, train_start_date, test_start_date, test_end_date,
                    _iter_, cfg.ITER_LSTM, cfg.EPOCH_LSTM, desired_date)
opts = make_opts(cfg, data, metadata, target_variable)
rows = make_rows(spec_list, rolling_selections, done_specs)
print(f"{len(rows)} specs queued ({len(done_specs)} already done); {cfg.MAX_WORKERS} workers; saving in {performance_file}")

# 3) run and collect
perf_out, fcst_out, oos_out, loo_out, failed = [], [], [], [], []
def _save():
    if perf_out:
        pd.concat([performance_table] + perf_out, axis=0).to_excel(f"{PATH_OUTPUT}/{performance_file}")
    if fcst_out:
        _t = pd.concat([result_table] + fcst_out, axis=0); _t.index = pd.to_datetime(_t.index).strftime("%Y-%m-%d")
        _t.to_excel(f"{PATH_OUTPUT}/{forecast_file}", index_label="date")
    if oos_out:
        _t = pd.concat([outofsample_table] + oos_out, axis=0); _t.index = pd.to_datetime(_t.index).strftime("%Y-%m-%d")
        _t.to_excel(f"{PATH_OUTPUT}/{outofsample_file}", index_label="date")

with tqdm(total=len(rows), desc="specs") as pbar:
    for n, (spec, perf_df, fcst_df, oos_df, loo_list, err) in enumerate(
            Parallel(n_jobs=cfg.MAX_WORKERS, backend="loky", return_as="generator")(
                delayed(run_one_spec)(r, opts, ctx) for r in rows), start=1):
        pbar.update(1)
        if err is not None:
            failed.append(spec); print(f"FAILED spec={spec}\n{err}\n"); continue
        if perf_df is not None and not perf_df.empty: perf_out.append(perf_df)
        if fcst_df is not None and not fcst_df.empty: fcst_out.append(fcst_df)
        if oos_df  is not None and not oos_df.empty:  oos_out.append(oos_df)
        if loo_list: loo_out.extend(loo_list)
        if n % cfg.SAVE_BATCH_SIZE == 0: _save(); print(f"saved after {n} specs")

# 4) final tables
if perf_out: performance_table = pd.concat([performance_table] + perf_out, axis=0)
if fcst_out: result_table      = pd.concat([result_table] + fcst_out, axis=0)
if oos_out:  outofsample_table = pd.concat([outofsample_table] + oos_out, axis=0)
performance_table.to_excel(f"{PATH_OUTPUT}/{performance_file}")
for _t, _f in ((result_table, forecast_file), (outofsample_table, outofsample_file)):
    _t.index = pd.to_datetime(pd.to_datetime(_t.index).strftime("%Y-%m-%d"))
    _t.to_excel(f"{PATH_OUTPUT}/{_f}", index_label="date")
shock_residual_file = f"{date_str} 3_nwcst Tab shock_residuals.xlsx"   # read by the watchlist cell (shock-robustness panel)
if loo_out:
    pd.DataFrame(loo_out).to_excel(f"{PATH_OUTPUT}/{shock_residual_file}", index=False)
    print(f"Wrote {len(loo_out)} leave-one-out shock residuals to {shock_residual_file}")
print(f"\nTotal runtime: {timeit.default_timer() - start:.0f}s; failed specs: {failed or 'none'}")
print(performance_table["RMSE"].describe()); print(f"Original Benchmark: {benchmark}")
performance_table


## 8. Watchlist, regime-aware nowcast, dispersion bands

On the pooled readings of this run (or of the merged seeds, when run by `run_tto.py`): A. the watchlist clusters and their alerts; B. the regime per (quarter, vintage) and the inverse-RMSE weights on the known prints; C. the weighted percentile bands.


In [ ]:
# [tto:post]
# Post-estimation: A. watchlist, B. regime-aware nowcast, C. dispersion bands.
# Reads:  the fcst and shock_residuals tables of this run (PATH_OUTPUT); pool_mask, reading_guard, VINTAGES (data cell);
#         pre_selected_columns_monthly, broken_vars, SHOCK_SHARE (pre-selection cells); spec_list; cfg section 6.
# Writes: <date> 4_rslt watchlist.xlsx, regime_aware_nowcast.xlsx, dispersion_bands.xlsx; in the namespace:
#         watch_summary, regime, ra, bands, WEIGHT_CACHE and the functions the live cell reuses
#         (breadth_at, known_actual_cutoff, weights_for, _weighted_quantile, _family).
# =========================================================
# A. Watchlist
# Panel 1: P25/P50/P75 of the pooled readings per (date, Vintage)
# Panel 2: cluster medians vs the pooled median; alert = divergence <= -ALERT_PP (ALERT_SIDE below) or |divergence| >= ALERT_PP (both)
# Panel 3: shock-robustness score per (spec, estimator) from the COVID LOO residuals
# =========================================================
import pandas as pd, numpy as np, re

fcst_w = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", parse_dates=["date"])
fcst_w = reading_guard(fcst_w[pool_mask(fcst_w)], label="watchlist")   # pool filter + reading guard (data cell)
long_w = fcst_w.melt(id_vars=["date", "spec", "estimator", "actuals"],
                     value_vars=list(VINTAGES), var_name="horizon", value_name="pred")   # VINTAGES: data cell
long_w["Vintage"] = long_w["horizon"].map(VINTAGES)
long_w = long_w.dropna(subset=["pred"])

# ---- Panel 1: pooled spread ----
spread = long_w.groupby(["date", "Vintage"]).agg(
    n=("pred", "size"),
    p25=("pred", lambda s: s.quantile(0.25)),
    p50=("pred", "median"),
    p75=("pred", lambda s: s.quantile(0.75)),
    actual=("actuals", "first"),
).reset_index().round(3)

# ---- Panel 2: clusters ----
# Spec names are <method><K><variant>; bench<K> is the base variant of the best method -> resolve from spec_list.
_bench_method = {r.spec: r.method for r in spec_list.itertuples() if str(r.spec).startswith("bench")}
def _family(s):
    m = re.match(r"^(f?(?:lars|sis|tstat|shap|ocmt|ibma)|common|shared|bench|human)(\d*)(flats|p5l2|p5|p3)?$", s)   # f-prefix: frozen groups
    if not m:
        return None, None, None
    meth = _bench_method.get(s, m.group(1)) if m.group(1) == "bench" else m.group(1)
    return meth, (int(m.group(2)) if m.group(2) else None), (m.group(3) or "base")

FACTOR, RAW = {"p3", "p5", "p5l2"}, {"base", "flats"}
# Cluster definitions live in config section 6 (WATCHLIST), filled from the identification run.
# name: (vintage or None = all, methods, variants, estimators or None = all, (K_from, K_to), note)
WATCHLIST = {k: (v["vintage"], set(v["methods"]), set(v["variants"]), (set(v["estimators"]) if v["estimators"] else None),
                 tuple(v["K"]), v["note"], bool(v.get("alerts", True))) for k, v in cfg.WATCHLIST.items()}
if not WATCHLIST:
    print("[watchlist] no clusters defined in config.WATCHLIST: the watchlist panel is empty; the regime can switch only on the "
          f"breadth trigger ({'enabled' if cfg.BREADTH_TRIGGER else 'disabled'})")
long_w["method"], long_w["K"], long_w["variant"] = zip(*long_w["spec"].apply(_family))
pooled_w = long_w.groupby(["date", "Vintage"])["pred"].median().rename("pooled_median")

panels, _summary = {}, []
for name, (V, meths, vars_, ests, (klo, khi), note, can_alert) in WATCHLIST.items():
    m = long_w["method"].isin(meths) & long_w["variant"].isin(vars_)
    if ests is not None:
        m &= long_w["estimator"].isin(ests)
    if klo is not None:
        m &= long_w["K"].between(klo, khi)
    if V is not None:
        m &= long_w["Vintage"].eq(V)
    g = long_w[m]
    if g.empty:
        print(f"[watchlist] {name}: no cells match -> skipped"); continue
    p = g.groupby(["date", "Vintage"]).agg(n=("pred", "size"),
                                           p25=("pred", lambda s: s.quantile(0.25)),
                                           median=("pred", "median"),
                                           p75=("pred", lambda s: s.quantile(0.75)),
                                           actual=("actuals", "first")).join(pooled_w)
    p["divergence"] = p["median"] - p["pooled_median"]
    # ALERT_SIDE "below" = crash alarm (cluster median at least ALERT_PP below the pool), "both" = either side.
    # flag = side of a divergence beyond ALERT_PP (-1 below, +1 above, 0 inside); only the ALERT_SIDE side becomes an
    # alert (the one thing that can switch the regime). Upward flags are information only.
    p["flag"] = np.where(p["divergence"] <= -cfg.ALERT_PP, -1, np.where(p["divergence"] >= cfg.ALERT_PP, 1, 0)) * int(can_alert)
    _div_hit = (p["flag"] == -1) if cfg.ALERT_SIDE == "below" else (p["flag"] != 0)
    p["alert"] = _div_hit.astype(int)
    p["cluster"] = name
    panels[name] = p.reset_index().round(3)
    _summary.append(p.reset_index().assign(cluster=name)[["date", "Vintage", "cluster", "median", "pooled_median", "divergence", "flag", "alert", "actual"]])
watch_summary = (pd.concat(_summary).sort_values(["date", "Vintage", "cluster"]).round(3) if _summary
                 else pd.DataFrame(columns=["date", "Vintage", "cluster", "median", "pooled_median", "divergence", "flag", "alert", "actual"]))
_alerts = watch_summary[watch_summary["alert"] == 1]
print(f"Watchlist alerts (cluster median - pooled median <= -{cfg.ALERT_PP:g} pp):" if cfg.ALERT_SIDE == "below" else f"Watchlist alerts (|cluster median - pooled median| >= {cfg.ALERT_PP:g} pp):")
print(_alerts.to_string(index=False) if len(_alerts) else "  none")
_flags_up = watch_summary[watch_summary["flag"] == 1]
print(f"Upward flags (cluster median - pooled median >= +{cfg.ALERT_PP:g} pp, information only, never switch the regime):")
print(_flags_up.to_string(index=False) if len(_flags_up) else "  none")

# ---- Panel 3: shock-robustness score (COVID LOO residuals) ----
shock_score = None
try:
    loo_w = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 3_nwcst Tab shock_residuals.xlsx")
    shock_score = loo_w[["spec", "estimator", "residual"]].copy()
    shock_score["shock_score_abs"] = shock_score["residual"].abs()
    shock_score = shock_score.sort_values("shock_score_abs")
except Exception as e:
    print(f"[watchlist] no shock_residuals.xlsx ({e}) - shock-score panel skipped")

watch_out = f"{PATH_OUTPUT}/{date_str} 4_rslt watchlist.xlsx"
with pd.ExcelWriter(watch_out) as xw:
    spread.to_excel(xw, sheet_name="spread", index=False)
    watch_summary.to_excel(xw, sheet_name="watchlist_summary", index=False)
    for name, p in panels.items():
        p.to_excel(xw, sheet_name=name[:31], index=False)
    pd.DataFrame([{"cluster": k, "vintage": v[0] if v[0] is not None else "all", "methods": ",".join(sorted(v[1])),
                   "variants": ",".join(sorted(v[2])), "estimators": ",".join(sorted(v[3])) if v[3] else "all",
                   "K_from": v[4][0], "K_to": v[4][1], "note": v[5], "alerts": v[6]} for k, v in WATCHLIST.items()]
                 ).to_excel(xw, sheet_name="definitions", index=False)
    if shock_score is not None:
        shock_score.round(3).to_excel(xw, sheet_name="shock_robustness", index=False)
print(f"[OK] Wrote watchlist to {watch_out}")

# =========================================================
# B. Regime-aware nowcast: automatic, real-time regime; expanding-window weights
#  1. REGIME. A quarter q is in the shock regime at vintage v if, on the information set of that vintage,
#     the breadth rule fires in any observed month of q: >= max(4, SHOCK_SHARE * n) of the n monthly
#     variables carrying shock features (guard-cleaned) sit below -3 sigma of their training distribution.
#     Publication lags from the metadata decide which months are observed. No typed dates; an explicit,
#     logged override (REGIME_OVERRIDE) replaces the analyst's silent list.
#  2. WEIGHTS. Inverse squared RMSE per (spec, estimator, vintage), computed only on test quarters whose
#     official GDP print was available at the reading date (release lag GDP_RELEASE_LAG months):
#     ordinary weights from past ordinary quarters, shock weights from past shock quarters.
#  3. FALLBACKS, stated: no past ordinary residual yet -> equal weights. The shock record starts with the pandemic
#     leave-one-out residuals, so a flagged quarter uses shock weights from the first shock on; only a
#     missing shock_residuals file sends it back to ordinary weights.
# =========================================================
import pandas as pd, numpy as np

GDP_RELEASE_LAG = int(metadata.set_index("ticket").loc[target_variable, "months_lag"])   # from the metadata (Trinidad: 6)
print(f"[regime] GDP release lag from metadata: {GDP_RELEASE_LAG} months")
BURN_IN_MIN_PRINTS = cfg.BURN_IN_MIN_PRINTS   # 3: ordinary weights need at least 3 published prints on record; before that every cell gets the same weight
REGIME_OVERRIDE = {"add": dict(cfg.REGIME_OVERRIDE_ADD), "remove": dict(cfg.REGIME_OVERRIDE_REMOVE)}   # analyst overrides, logged with their reason

fcst_ra = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", parse_dates=["date"])
fcst_ra = reading_guard(fcst_ra[pool_mask(fcst_ra)], label="regime")   # pool filter + reading guard (data cell)

# ---- pandemic on the shock record: one leave-one-out residual per cell and seed for 2020-Q2 ----
try:
    _loo = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 3_nwcst Tab shock_residuals.xlsx")
    _loo = _loo[pool_mask(_loo)][["spec", "estimator", "residual"]].rename(columns={"residual": "resid"})
    _cells = fcst_ra[["spec", "estimator"]].drop_duplicates()
    _missing = _cells.merge(_loo[["spec", "estimator"]].drop_duplicates(), how="left", indicator=True).query("_merge == 'left_only'")[["spec", "estimator"]]
    _loo_med = float(_loo.groupby(["spec", "estimator"])["resid"].apply(lambda x: np.sqrt(np.mean(x ** 2))).median())
    PANDEMIC_RECORD = pd.concat([_loo, _missing.assign(resid=_loo_med)], ignore_index=True)   # cells without a refit (LSTM) sit at the median
    print(f"[regime] pandemic record: {_loo[['spec', 'estimator']].drop_duplicates().shape[0]} cells from leave-one-out, "
          f"{len(_missing)} cells at the median residual {_loo_med:.1f}")
except FileNotFoundError:
    PANDEMIC_RECORD = None; print("[regime] no shock_residuals file: shock quarters fall back to ordinary weights")
# VINTAGES comes from the data cell (any vintage grid)
long_df = fcst_ra.melt(id_vars=["date", "spec", "estimator", "actuals"], value_vars=list(VINTAGES),
                       var_name="horizon", value_name="pred")
long_df["Vintage"] = long_df["horizon"].map(VINTAGES)
long_df = long_df.dropna(subset=["pred"])
long_df["resid"] = long_df["pred"] - long_df["actuals"]

# ---- 1. real-time breadth indicator per (quarter, vintage) ----
# Fixed panel: every guard-cleaned monthly candidate (not only the variables the screens selected), so the
# regime does not change with the selection. Standardisation against the training distribution of the
# QUARTERLY MINIMA, exactly as the SHOK0000 feature in the shock-features cell.
_vars = [v for v in pre_selected_columns_monthly if v in data.columns and v not in broken_vars]
_lag = metadata.set_index("ticket")["months_lag"].reindex(_vars).fillna(1).astype(int)
_qmin_tr = data.loc[data.index < test_start_date, _vars].resample("QS").min()
_mu, _sg = _qmin_tr.mean(), _qmin_tr.std().replace(0, np.nan)
_thr = max(4, int(np.ceil(SHOCK_SHARE * len(_vars))))

def breadth_at(q_end, v):
    """Number of variables whose within-quarter minimum (over the months observed at vintage v) is below -3 sigma."""
    q_start = q_end - pd.DateOffset(months=2); reading = q_end + pd.DateOffset(months=v)
    cnt = 0
    for var in _vars:
        last_obs = reading - pd.DateOffset(months=int(_lag[var]))          # latest month published at the reading date
        obs = data.loc[q_start:min(q_end, last_obs), var].dropna()
        if len(obs) and (obs.min() - _mu[var]) / _sg[var] < -3:
            cnt += 1
    return cnt

test_q = sorted(long_df["date"].unique())
regime = pd.DataFrame([{"date": q, "Vintage": v, "breadth": breadth_at(pd.Timestamp(q), v)}
                       for q in test_q for v in VINTAGES.values()])
# ---- trigger: breadth threshold, OR a watchlist alert corroborated by broken variables ----
# With BREADTH_TRIGGER = False in config, only the corroborated watchlist alert switches the regime.
# The watchlist (block A) is specific and early for the kind of shock it was identified on and blind to others; the
# breadth count is generic but late. Neither suffices alone, so the regime switches on either, and a watchlist alert
# counts only if at least MIN_BROKEN_FOR_ALERT variables in the panel are broken at that vintage. That minimum is
# max(3, floor(threshold/3)), so it scales with the panel (3 with a threshold of 10, 5 with a threshold of 15). Once
# triggered at a vintage, the quarter stays in the shock regime at the later vintages of that quarter.
MIN_BROKEN_FOR_ALERT = max(3, _thr // 3)
_wa = watch_summary[watch_summary["alert"] == 1][["date", "Vintage"]].drop_duplicates().assign(watch_alert=True)
regime = regime.merge(_wa, on=["date", "Vintage"], how="left"); regime["watch_alert"] = regime["watch_alert"].fillna(False).astype(bool)
regime["trig_breadth"] = (regime["breadth"] >= _thr) & cfg.BREADTH_TRIGGER   # False = corroboration only
regime["trig_watch"] = regime["watch_alert"] & (regime["breadth"] >= MIN_BROKEN_FOR_ALERT)
regime = regime.sort_values(["date", "Vintage"])
regime["is_shock"] = regime.groupby("date")[["trig_breadth", "trig_watch"]].transform("cummax").any(axis=1)
regime["trigger"] = np.select([regime.trig_breadth & regime.trig_watch, regime.trig_breadth, regime.trig_watch], ["both", "breadth", "watchlist"], "")
print(f"[regime] watchlist alerts need >= {MIN_BROKEN_FOR_ALERT} broken variables; triggers by (quarter, vintage):")
print(regime[regime["trigger"] != ""][["date", "Vintage", "breadth", "trigger"]].to_string(index=False) if (regime["trigger"] != "").any() else "  none")
for d_, why in REGIME_OVERRIDE["add"].items():    regime.loc[regime["date"] == pd.Timestamp(d_), "is_shock"] = True
for d_, why in REGIME_OVERRIDE["remove"].items(): regime.loc[regime["date"] == pd.Timestamp(d_), "is_shock"] = False
regime["override"] = regime["date"].dt.strftime("%Y-%m-%d").map({**REGIME_OVERRIDE["add"], **REGIME_OVERRIDE["remove"]}).fillna("")
regime["threshold"], regime["n_panel"] = _thr, len(_vars)      # read by the chart cells
print(f"[regime] breadth threshold {_thr} of {len(_vars)}; shock-regime (quarter, vintage) cells: {int(regime['is_shock'].sum())}")

# ---- 2. expanding-window weights ----
long_df = long_df.merge(regime[["date", "Vintage", "is_shock"]], on=["date", "Vintage"])
long_df["q_end"] = long_df["date"]                 # quarter-end month (dates are Mar/Jun/Sep/Dec)

def known_actual_cutoff(q_end, v):
    """Quarters whose official print is available at the reading date q_end + v months."""
    return q_end + pd.DateOffset(months=v) - pd.DateOffset(months=GDP_RELEASE_LAG)
regime["prints_known"] = [sum(pd.Timestamp(d) <= known_actual_cutoff(pd.Timestamp(q), v) for d in test_q)
                          for q, v in zip(regime["date"], regime["Vintage"])]      # prints behind the weights
regime["burn_in"] = regime["prints_known"] < BURN_IN_MIN_PRINTS                     # equal weights in these cells; the chart hatches them

def weights_for(q_end, v, shock):
    cut = known_actual_cutoff(q_end, v)
    hist = long_df[(long_df["Vintage"] == v) & (long_df["q_end"] <= cut) & (long_df["is_shock"] == shock)][["spec", "estimator", "resid"]]
    if not shock and sum(pd.Timestamp(d) <= cut for d in test_q) < BURN_IN_MIN_PRINTS:
        return None                                   # burn-in: a record of 1-2 prints is a lottery, not a ranking -> equal weights
    if shock and PANDEMIC_RECORD is not None:
        hist = pd.concat([hist, PANDEMIC_RECORD], ignore_index=True)    # the pandemic is on the record at every vintage
    if hist.empty:
        return None
    rmse = hist.groupby(["spec", "estimator"])["resid"].apply(lambda x: np.sqrt(np.nanmean(x ** 2)))
    w = (1.0 / rmse ** 2).replace([np.inf, -np.inf], np.nan).fillna(0)
    return w / w.sum() if w.sum() > 0 else None

out_rows, WEIGHT_CACHE = [], {}
for (q, v), g in long_df.groupby(["date", "Vintage"]):
    shock = bool(g["is_shock"].iloc[0]); q_end = pd.Timestamp(q)
    w = weights_for(q_end, v, shock=True) if shock else None
    shock_w_available = w is not None
    if w is None:
        w = weights_for(q_end, v, shock=False)          # ordinary weights (or shock regime without shock history)
    if w is None:                                        # first test quarter: no residual history at all
        pred = float(g["pred"].mean()); scheme = "equal"
    else:
        wk = np.array([w.get((s, e), 0.0) for s, e in zip(g["spec"], g["estimator"])])
        pred = float((wk * g["pred"].values).sum() / wk.sum()) if wk.sum() > 0 else float(g["pred"].mean())
        scheme = "shock" if (shock and shock_w_available) else "ordinary"
    WEIGHT_CACHE[(q_end, v)] = w
    out_rows.append({"date": q_end, "Vintage": v, "is_shock": shock, "breadth": int(g.merge(regime, on=["date","Vintage"])["breadth"].iloc[0]),
                     "weights": scheme, "actual": g["actuals"].iloc[0], "regime_aware_nowcast": pred})
ra = pd.DataFrame(out_rows); ra["error"] = ra["regime_aware_nowcast"] - ra["actual"]
with pd.ExcelWriter(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx") as xw:
    ra.to_excel(xw, sheet_name="nowcast", index=False)
    regime.to_excel(xw, sheet_name="regime_log", index=False)
print(f"[OK] regime-aware nowcast written; weight schemes used: {ra['weights'].value_counts().to_dict()}")
print(ra[ra["is_shock"]][["date", "Vintage", "breadth", "weights", "regime_aware_nowcast", "error"]].round(2).to_string(index=False) if ra["is_shock"].any() else "[regime] no shock-regime quarter in the test window")

# =========================================================
# C. Dispersion bands: regime-conditional inverse-RMSE weighted quantiles of the (spec, estimator, seed) pool
# =========================================================
import pandas as pd, numpy as np

fcst_d = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", parse_dates=["date"])
fcst_d = reading_guard(fcst_d[pool_mask(fcst_d)], label="bands")   # pool filter + reading guard (data cell)
ra_d   = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx", sheet_name="nowcast", parse_dates=["date"])

# The bands use the regime and the expanding-window weights of block B (WEIGHT_CACHE),
# so consensus and bands are always computed under the same regime and the same information set.
VINTAGES_D = dict(VINTAGES)      # data cell (any vintage grid)
id_vars = ["date", "spec", "estimator", "actuals"] + (["seed"] if "seed" in fcst_d.columns else [])
long_d = fcst_d.melt(id_vars=id_vars, value_vars=list(VINTAGES_D), var_name="horizon", value_name="pred")
long_d["Vintage"] = long_d["horizon"].map(VINTAGES_D)
long_d = long_d.dropna(subset=["pred"])

def _weighted_quantile(values, weights, q):
    """Weighted percentile, q in [0, 1]. Sort by value, cum-sum weights, interpolate."""
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & (weights > 0)
    if mask.sum() == 0:
        return np.nan
    v, w = values[mask], weights[mask]
    order = np.argsort(v)
    v, w = v[order], w[order]
    cw = np.cumsum(w)
    cw /= cw[-1]
    return float(np.interp(q, cw, v))

Q_LEVELS = {"p025": 0.025, "p10": 0.10, "p25": 0.25, "p50": 0.50,
            "p75": 0.75, "p90": 0.90, "p975": 0.975}

rows_d = []
for (date, V), g in long_d.groupby(["date", "Vintage"]):
    w = WEIGHT_CACHE.get((pd.Timestamp(date), V))
    weights = np.ones(len(g)) if w is None else np.array([w.get((s, e), 0.0) for s, e in zip(g["spec"], g["estimator"])])
    preds = g["pred"].values.astype(float)
    row = {"date": date, "Vintage": V, "is_shock": bool(ra.loc[(ra["date"] == pd.Timestamp(date)) & (ra["Vintage"] == V), "is_shock"].iloc[0]),
           "actual": g["actuals"].iloc[0]}
    for lbl, q in Q_LEVELS.items():
        row[lbl] = _weighted_quantile(preds, weights, q)
    rows_d.append(row)

bands = pd.DataFrame(rows_d).merge(
    ra_d[["date", "Vintage", "regime_aware_nowcast"]], on=["date", "Vintage"], how="left"
)

# Aliases read by the chart notebook (4_results_tto.ipynb)
bands["lo_80"] = bands["p10"]
bands["hi_80"] = bands["p90"]
bands["lo_95"] = bands["p025"]
bands["hi_95"] = bands["p975"]

out_cols = ["date", "Vintage", "actual", "regime_aware_nowcast", "is_shock",
            "p025", "p10", "p25", "p50", "p75", "p90", "p975",
            "lo_80", "hi_80", "lo_95", "hi_95"]
out_path_d = f"{PATH_OUTPUT}/{date_str} 4_rslt dispersion_bands.xlsx"
bands[out_cols].to_excel(out_path_d, index=False)
print(f"[OK] Wrote {len(bands)} rows to {out_path_d}")

_shock_q = bands[bands["is_shock"]]["date"].unique()
for _d in _shock_q:
    print(f"\n{pd.Timestamp(_d):%Y-%m} (shock regime) dispersion bands (weighted percentiles):")
    print(bands[bands["date"] == _d][["Vintage", "regime_aware_nowcast", "p025", "p10", "p50", "p90", "p975"]].round(2).to_string(index=False))


## 9. Live quarter

The unprinted quarters, each read at its own vintage, through the same regime, cluster and band machinery. Empty in the identification run.


In [ ]:
# [tto:live]
# Live quarter: the out-of-sample readings go through the same regime, cluster and band machinery as the backtest.
# Reads:  the oos table of this run; breadth_at, _thr, MIN_BROKEN_FOR_ALERT, weights_for, PANDEMIC_RECORD, WATCHLIST,
#         _family, _weighted_quantile, Q_LEVELS, REGIME_OVERRIDE (post cell); cfg section 6.
# Writes: <date> 4_rslt live_quarter.xlsx (sheets nowcast, clusters), one row per unprinted quarter.
# One reading date: the data cut (last month with monthly data), which fixes the vintage. The LSTM has no
# out-of-sample reading, so the live pool is the sklearn cells only and the weights are renormalised over them.
# =========================================================
if desired_date is None:
    print('[live] no live quarter in this run (identification run)')
else:
    import pandas as pd, numpy as np

    oos = pd.read_excel(f"{PATH_OUTPUT}/{outofsample_file}", parse_dates=["date"])
    oos = reading_guard(oos[pool_mask(oos)].rename(columns={"nowcast": "pred"}), cols=["pred"], label="live").dropna(subset=["pred"])
    _monthly_cols = [c for c in data.columns if c != target_variable]
    data_cut = data.index[data[_monthly_cols].notna().any(axis=1)].max()           # last month with any monthly data
    # Every unprinted quarter in the out-of-sample table is read at its own vintage (with data to October 2026: Q2 at V+4
    # and Q3 at V+1); a vintage beyond the grid is capped at its last vintage. One row per quarter in the live file.
    _oos_all = oos.copy(); _live_rows, _live_cluster_tabs = [], []
    for live_date in sorted(pd.to_datetime(_oos_all["date"].unique())):
        if live_date <= pd.Timestamp(test_end_date):
            continue                                                               # published quarters are not live
        oos = _oos_all[_oos_all["date"] == live_date].copy()
        live_V = (data_cut.year - live_date.year) * 12 + (data_cut.month - live_date.month)
        _V_eff = min(live_V, max(lags))                                          # weights, record and clusters of the nearest backtested vintage
        live_q = f"{live_date.year}-Q{(live_date.month - 1) // 3 + 1}"
        print(f"[live] {live_q} read at V{live_V:+d} (data to {data_cut:%Y-%m}; record of V{_V_eff:+d}); {oos[['spec', 'estimator']].drop_duplicates().shape[0]} cells, {len(oos)} readings")

        # ---- regime at the live reading: breadth, clusters, trigger ----
        live_breadth = breadth_at(live_date, _V_eff)
        oos["method"], oos["K"], oos["variant"] = zip(*oos["spec"].apply(_family))
        live_pooled = float(oos["pred"].median())
        _rows = []
        for name, (V, meths, vars_, ests, (klo, khi), note, can_alert) in WATCHLIST.items():
            m = oos["method"].isin(meths) & oos["variant"].isin(vars_)
            if ests is not None: m &= oos["estimator"].isin(ests)
            if klo is not None:  m &= oos["K"].between(klo, khi)
            g = oos[m]
            if g.empty:
                print(f"[live] {name}: no cells in the live pool"); continue
            med = float(g["pred"].median())
            _rows.append({"date": live_date, "Vintage": live_V, "cluster": name, "n": int(len(g)), "median": med, "pooled_median": live_pooled,
                          "divergence": med - live_pooled,
                          "flag": (-1 if med - live_pooled <= -cfg.ALERT_PP else (1 if med - live_pooled >= cfg.ALERT_PP else 0)) * int(can_alert),
                          "alert": int(((med - live_pooled <= -cfg.ALERT_PP) if cfg.ALERT_SIDE == "below" else (abs(med - live_pooled) >= cfg.ALERT_PP)) and can_alert), "read_at": f"V{live_V:+d}", "defined_at": ("all" if V is None else f"V{V:+d}")})
        live_clusters = pd.DataFrame(_rows)
        live_alert = bool(live_clusters["alert"].any()) if len(live_clusters) else False
        trig_breadth = bool(cfg.BREADTH_TRIGGER) and (live_breadth >= _thr)
        trig_watch = live_alert and (live_breadth >= MIN_BROKEN_FOR_ALERT)
        live_shock = bool(trig_breadth or trig_watch)
        _k = live_date.strftime("%Y-%m-%d")
        if _k in REGIME_OVERRIDE["add"]:    live_shock = True
        if _k in REGIME_OVERRIDE["remove"]: live_shock = False
        live_trigger = "both" if (trig_breadth and trig_watch) else ("breadth" if trig_breadth else ("watchlist" if trig_watch else ""))

        # ---- weights at the live vintage (same function as the backtest), nowcast and bands ----
        w = weights_for(live_date, _V_eff, shock=True) if live_shock else None
        scheme = "shock" if w is not None else None
        if w is None:
            w = weights_for(live_date, _V_eff, shock=False); scheme = "ordinary" if w is not None else "equal"
        wk = np.ones(len(oos)) if w is None else np.array([w.get((s, e), 0.0) for s, e in zip(oos["spec"], oos["estimator"])])
        if wk.sum() == 0: wk = np.ones(len(oos)); scheme = "equal"
        live_nowcast = float((wk * oos["pred"].values).sum() / wk.sum())
        _q = {lbl: _weighted_quantile(oos["pred"].values.astype(float), wk, q) for lbl, q in Q_LEVELS.items()}
        live_prints = sum(pd.Timestamp(d) <= known_actual_cutoff(live_date, _V_eff) for d in test_q)
        live_row = pd.DataFrame([{"date": live_date, "quarter": live_q, "Vintage": live_V, "data_cut": data_cut, "is_shock": live_shock, "breadth": int(live_breadth),
                                  "threshold": _thr, "n_panel": len(_vars), "trigger": live_trigger, "watch_alert": live_alert, "weights": scheme,
                                  "prints_known": int(live_prints), "n_cells": int(oos[["spec", "estimator"]].drop_duplicates().shape[0]),
                                  "regime_aware_nowcast": live_nowcast, "pooled_median": live_pooled,
                                  "lo_80": _q["p10"], "hi_80": _q["p90"], "lo_95": _q["p025"], "hi_95": _q["p975"]}])
        _live_rows.append(live_row); _live_cluster_tabs.append(live_clusters)
        print(f"[live] breadth {live_breadth} of {len(_vars)} (threshold {_thr}); cluster alert {live_alert}; regime {'SHOCK' if live_shock else 'ordinary'}"
              f"{' (' + live_trigger + ')' if live_trigger else ''}; weights {scheme} on {live_prints} prints")
        print(f"[live] regime-aware nowcast {live_nowcast:+.2f}; pooled median {live_pooled:+.2f}; 80% band [{_q['p10']:+.2f}, {_q['p90']:+.2f}]; 95% band [{_q['p025']:+.2f}, {_q['p975']:+.2f}]")
        print(live_clusters[["cluster", "n", "median", "pooled_median", "divergence", "flag", "alert", "read_at", "defined_at"]].round(2).to_string(index=False))
    if _live_rows:
        with pd.ExcelWriter(f"{PATH_OUTPUT}/{date_str} 4_rslt live_quarter.xlsx") as xw:
            pd.concat(_live_rows, ignore_index=True).to_excel(xw, sheet_name="nowcast", index=False)
            pd.concat(_live_cluster_tabs, ignore_index=True).to_excel(xw, sheet_name="clusters", index=False)
        print(f"[live] {len(_live_rows)} live quarter(s) written")
    else:
        print("[live] no unprinted quarter in the out-of-sample table")
